# 🔧 SageMaker DevOps Troubleshooting Workshop

**An alert comes in.** The ML platform that serves thousands of students has encountered issues. Training jobs are failing. Endpoints are unhealthy. Auto-scaling needs attention.

This workshop provides hands-on experience troubleshooting real production scenarios. You'll learn how to investigate and resolve common ML workflow issues using AWS DevOps Agent.

**Your Mission:** Work through three troubleshooting scenarios, using AWS DevOps Agent to investigate and resolve each issue.

**Let's begin.**

---

# Troubleshooting SageMaker AI Issues with AWS DevOps Agent - Troubleshooting Workshop
### Objective : Learn to troubleshoot AI/ML workflows by intentionally breaking and fixing a SageMaker training job and amodel deployment using AWS DevOps Agent.


## Workshop Overview
| Part | Scenario | Issue Type |
|------|----------|------------|
| 1 | Setup & Data Generation | Setup |
| 2-3 | SageMaker Training Script Failure | KeyError issue |
| 4-5 | Hosting SageMaker Endpoint Unhealthy Container | Missing ping + memory leak |
| 6-7 | Auto-scaling Chaos | Aggressive scaling |
| 8 | Cleanup | Resource cleanup |

## 📖 The Story Behind This Workshop

In production ML systems, issues can occur at any time and require rapid diagnosis across logs, metrics, code, and infrastructure.

Traditional troubleshooting means:
- Manually searching through thousands of log lines
- Correlating metrics across multiple dashboards  
- SSHing into containers to debug
- Hoping someone remembers the last time this happened

**AWS DevOps Agent changes the game.** It's an AI-powered first responder that:
- Continuously monitors CloudWatch signals
- Automatically detects anomalies
- Analyzes code, logs, and infrastructure
- Provides actionable recommendations

Today, you'll see the difference firsthand.

---

## Part 1: Setup

In [ ]:
import boto3
import sagemaker
import pandas as pd
import numpy as np
from sagemaker import get_execution_role
from sagemaker.inputs import TrainingInput
from sagemaker.estimator import Estimator
import json
import time

# Initialize SageMaker session
sagemaker_session = sagemaker.Session()
role = get_execution_role()
region = boto3.Session().region_name
bucket = sagemaker_session.default_bucket()
prefix = 'student-completion-workshop'

print(f"SageMaker Role: {role}")
print(f"Region: {region}")
print(f"Bucket: {bucket}")

In [ ]:
# Generate synthetic student data
np.random.seed(42)
n_samples = 5000

df = pd.DataFrame({
    'student_id': range(1, n_samples + 1),
    'courses_enrolled': np.random.randint(1, 10, n_samples),
    'courses_completed_prev': np.random.randint(0, 8, n_samples),
    'avg_time_per_module': np.random.uniform(10, 120, n_samples),
    'login_frequency': np.random.uniform(0.5, 14, n_samples),
    'forum_posts': np.random.randint(0, 50, n_samples),
    'assignment_submission_rate': np.random.uniform(0, 1, n_samples),
    'video_completion_rate': np.random.uniform(0, 1, n_samples),
    'days_since_last_login': np.random.randint(0, 60, n_samples)
})

completion_prob = (
    0.3 * df['assignment_submission_rate'] +
    0.25 * df['video_completion_rate'] +
    0.15 * (df['login_frequency'] / 14) +
    0.1 * (df['forum_posts'] / 50) +
    0.1 * (1 - df['days_since_last_login'] / 60) +
    0.1 * (df['courses_completed_prev'] / 8)
) + np.random.normal(0, 0.1, n_samples)

df['completed'] = (completion_prob > 0.5).astype(int)
print(f'Dataset shape: {df.shape}')
print(f'Completion rate: {df["completed"].mean():.2%}')
df.head()

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)
train_df.to_csv('train.csv', index=False)
val_df.to_csv('validation.csv', index=False)

train_s3 = sagemaker_session.upload_data('train.csv', bucket=bucket, key_prefix=f'{prefix}/data')
val_s3 = sagemaker_session.upload_data('validation.csv', bucket=bucket, key_prefix=f'{prefix}/data')
print(f'Train: {train_s3}')
print(f'Val: {val_s3}')

In [ ]:
!mkdir -p scripts

## Part 2: Buggy Training Script

---

# 🚨 Scenario 1: The Training Job Mystery
## Status: Investigation Required

**The Alert:**
```
ALARM: SageMaker Training Job Failed
Job Name: student-completion-predictor-training
Status: Failed
Error: AlgorithmError
```

**The Situation:**
Your student course completion prediction model just failed mid-training. The job ran for 15 minutes, consumed compute resources, and then crashed with a cryptic error buried in CloudWatch Logs.

**The Impact:**
- Wasted compute costs
- Delayed model deployment
- Potential revenue loss if this isn't fixed quickly

**Your Challenge:**
Find out what went wrong and fix it. Fast.

**Traditional Approach:** Extended time for log searching, local reproduction, and debugging.

**With AWS DevOps Agent:** Let's see how much faster we can resolve this...

---

In [ ]:
%%writefile scripts/errortrain.py
# Training script with BUG: Wrong column name 'target' instead of 'completed'
import argparse
import os
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

def train():
    parser = argparse.ArgumentParser()
    parser.add_argument('--n-estimators', type=int, default=100)
    parser.add_argument('--max-depth', type=int, default=10)
    parser.add_argument('--model-dir', type=str, default=os.environ.get('SM_MODEL_DIR'))
    parser.add_argument('--train', type=str, default=os.environ.get('SM_CHANNEL_TRAIN'))
    parser.add_argument('--validation', type=str, default=os.environ.get('SM_CHANNEL_VALIDATION'))
    args = parser.parse_args()
    
    print('Loading training data...')
    train_df = pd.read_csv(os.path.join(args.train, 'train.csv'))
    val_df = pd.read_csv(os.path.join(args.validation, 'validation.csv'))
    
    # BUG: Column is named 'completed' but we're looking for 'target'
    # This will cause KeyError!
    feature_cols = ['courses_enrolled', 'courses_completed_prev', 'avg_time_per_module',
                    'login_frequency', 'forum_posts', 'assignment_submission_rate',
                    'video_completion_rate', 'days_since_last_login']
    
    X_train = train_df[feature_cols]
    y_train = train_df['target']  # BUG: Should be 'completed'
    X_val = val_df[feature_cols]
    y_val = val_df['target']  # BUG: Should be 'completed'
    
    print(f'Training samples: {len(X_train)}')
    print(f'Validation samples: {len(X_val)}')
    
    model = RandomForestClassifier(
        n_estimators=args.n_estimators,
        max_depth=args.max_depth,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    
    val_preds = model.predict(X_val)
    val_proba = model.predict_proba(X_val)[:, 1]
    print(f'Validation Accuracy: {accuracy_score(y_val, val_preds):.4f}')
    print(f'Validation AUC: {roc_auc_score(y_val, val_proba):.4f}')
    
    joblib.dump(model, os.path.join(args.model_dir, 'model.joblib'))
    print('Model saved!')

if __name__ == '__main__':
    train()

## Part 3: Run SageMaker Training Job (Will Fail)

In [ ]:
#The training job would fail with errors and we can continue to troubleshoot using the AWS DevOps Agent.

from sagemaker.sklearn.estimator import SKLearn

sklearn_estimator = SKLearn(
    entry_point='errortrain.py',
    source_dir='scripts',
    role=role,
    instance_count=1,
    instance_type='ml.m5.large',
    framework_version='1.0-1',
    py_version='py3',
    hyperparameters={
        'n-estimators': 100,
        'max-depth': 10
    },
    sagemaker_session=sagemaker_session,
    base_job_name='student-completion'
)

# THIS WILL FAIL - KeyError: 'target'
sklearn_estimator.fit({'train': train_s3, 'validation': val_s3}, wait=True, logs='All')

## Now Let us troubleshoot the Algorithm errors while running a SageMaker Training Job by using AWS DevOps Agent.
#### Navigate to the DevOps Agents Incident Response Dashboard and start the investigation by providing the following details about the issue you are observing.

###### Investigation Details: ```we are seeing an AlgorithmError while training a machine learning model student-completion SageMaker training job in our account```

###### Investigation Starting point : ```Please review the training container logs and the respective training job related configurations to identify the root cause of the issue.```


### Fix 1: Corrected Training Script

In [ ]:
%%writefile scripts/train.py
# FIXED: Changed 'target' to 'completed'
import argparse, os
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--n-estimators', type=int, default=100)
    parser.add_argument('--max-depth', type=int, default=10)
    parser.add_argument('--model-dir', type=str, default=os.environ.get('SM_MODEL_DIR'))
    parser.add_argument('--train', type=str, default=os.environ.get('SM_CHANNEL_TRAIN'))
    parser.add_argument('--validation', type=str, default=os.environ.get('SM_CHANNEL_VALIDATION'))
    args = parser.parse_args()
    
    train_df = pd.read_csv(os.path.join(args.train, 'train.csv'))
    val_df = pd.read_csv(os.path.join(args.validation, 'validation.csv'))
    
    features = ['courses_enrolled', 'courses_completed_prev', 'avg_time_per_module',
                'login_frequency', 'forum_posts', 'assignment_submission_rate',
                'video_completion_rate', 'days_since_last_login']
    
    X_train, y_train = train_df[features], train_df['completed']  # FIXED!
    X_val, y_val = val_df[features], val_df['completed']  # FIXED!
    
    model = RandomForestClassifier(n_estimators=args.n_estimators, max_depth=args.max_depth, random_state=42)
    model.fit(X_train, y_train)
    print(f'AUC: {roc_auc_score(y_val, model.predict_proba(X_val)[:,1]):.4f}')
    joblib.dump(model, os.path.join(args.model_dir, 'model.joblib'))

In [ ]:
# Re-run with fixed script
sklearn_estimator = SKLearn(
    entry_point='train.py',
    source_dir='scripts',
    role=role,
    instance_count=1,
    instance_type='ml.m5.large',
    framework_version='1.0-1',
    py_version='py3',
    hyperparameters={'n-estimators': 100, 'max-depth': 10},
    sagemaker_session=sagemaker_session,
    base_job_name='student-completion-fixed'
)
sklearn_estimator.fit({'train': train_s3, 'validation': val_s3}, wait=True, logs='All')
print('Training completed!')

## Part 4: Buggy Inference Script

---

# 🚨 Scenario 2: The Unhealthy Endpoint Nightmare
## Status: Investigation Required

**The Alert:**
```
ALARM: SageMaker Endpoint Health Check Failed
Endpoint Name: student-completion-predictor-endpoint
Status: Failed
Health Checks: 0/10 passing
Customer Impact: 503 Service Unavailable
```

**The Situation:**
The endpoint deployed successfully, but it's immediately failing health checks. Worse, when it does process requests, something is causing the container to crash. Customer-facing APIs are returning 503 errors.

**The Impact:**
- Service disruption for end users
- Failed health checks preventing traffic routing
- Potential memory issues causing container crashes
- Customer complaints starting to come in

**Your Challenge:**
The endpoint code has TWO critical bugs. Can you find them both?

**Traditional Approach:** Extended time debugging health checks and profiling memory issues.

**With AWS DevOps Agent:** Let's see how it identifies BOTH issues simultaneously...

---

In [ ]:
%%writefile scripts/errorinference.py
# BUGS: 1) Missing ping() 2) Memory leak 3) No error handling
import os, joblib
import pandas as pd
import numpy as np

_history = []  # MEMORY LEAK!

def model_fn(model_dir):
    return joblib.load(os.path.join(model_dir, 'model.joblib'))

# MISSING: def ping(): return ''

def input_fn(request_body, content_type):
    if content_type == 'text/csv':
        return pd.read_csv(pd.io.common.StringIO(request_body))
    raise Exception('Bad content type')  # Poor error msg

def predict_fn(data, model):
    global _history
    preds = model.predict_proba(data)[:, 1]
    _history.append(data.values.copy())  # LEAK!
    _history.append(preds.copy())  # LEAK!
    return preds

def output_fn(pred, accept):
    return ','.join(map(str, pred))

## Part 5: Deploy a SageMaker Endpoint (Will have issues due model/container issues)

In [ ]:
predictor = sklearn_estimator.deploy(
    initial_instance_count=1,
    instance_type='ml.t2.medium',
    endpoint_name='student-completion-endpoint',
    entry_point='errorinference.py',
    source_dir='scripts'
)
print(f'Endpoint: {predictor.endpoint_name}')

In [ ]:
# Test - may fail due to unhealthy container
test_data = val_df.drop(['student_id', 'completed'], axis=1).head(5)
try:
    resp = predictor.predict(test_data.to_csv(index=False), initial_args={'ContentType': 'text/csv'})
    print(f'Predictions: {resp}')
except Exception as e:
    print(f'Failed: {e}\nCheck CloudWatch logs!')

In [ ]:
# Test - may fail due to unhealthy container
test_data = val_df.drop(['student_id', 'completed'], axis=1).head(5)

max_retries = 5
retry_count = 0

while retry_count < max_retries:
    try:
        resp = predictor.predict(test_data.to_csv(index=False), initial_args={'ContentType': 'text/csv'})
        print(f'Predictions: {resp}')
        break  # Success - exit the loop
    except Exception as e:
        retry_count += 1
        print(f'Attempt {retry_count} failed: {e}')
        
        if retry_count >= max_retries:
            print(f'All {max_retries} attempts failed. Check CloudWatch logs!')
        else:
            print(f'Retrying... ({retry_count}/{max_retries})')

## Now Let us troubleshoot the Internal server error/ Model errors issue of the SageMaker Endpoint using AWS DevOps Agent.
#### Navigate to the DevOps Agents Incident Response Dashboard and start the investigation by providing the following details about the issue you are observing.

###### Investigation Details: ```we are seeing model invocation errors on the student-completion-endpoint SageMaker Endpoint in our account```

###### Investigation Starting point : ```Please review the model invocation logs and the respective endpoint related configurations to identify the root cause of the issue.```


### Fix 2: Corrected Inference Script

In [ ]:
%%writefile scripts/inference.py
# FIXED: Added ping(), removed memory leak, simple CSV handling
import os, joblib
import pandas as pd
from io import StringIO

def model_fn(model_dir):
    return joblib.load(os.path.join(model_dir, 'model.joblib'))

def ping():
    return ''

def input_fn(request_body, content_type):
    if content_type == 'text/csv':
        # Just read the CSV and use whatever columns come in
        # The model expects 8 numeric features in the correct order
        data = pd.read_csv(StringIO(request_body))
        return data
    raise ValueError(f'Unsupported content type: {content_type}')

def predict_fn(data, model):
    return model.predict_proba(data)[:, 1]

def output_fn(pred, accept):
    return ','.join(map(str, pred))

In [ ]:
# Delete old endpoint and redeploy
sm_client = boto3.client('sagemaker')
try:
    sm_client.delete_endpoint(EndpointName='student-completion-endpoint')
    print('Deleted old endpoint, waiting...')
    time.sleep(30)
    sm_client.delete_endpoint_config(EndpointConfigName='student-completion-endpoint')
    print('Deleted old endpoint config, waiting...')
except: pass

predictor = sklearn_estimator.deploy(
    initial_instance_count=1,
    instance_type='ml.m5.large',
    endpoint_name='student-completion-endpoint-fixed',
    entry_point='inference.py',
    source_dir='scripts'
)
print('Deployed with fixed inference script!')

In [ ]:
# Test the fixed endpoint using boto3 directly (avoids SDK serialization issues)
feature_cols = ['courses_enrolled', 'courses_completed_prev', 'avg_time_per_module',
                'login_frequency', 'forum_posts', 'assignment_submission_rate',
                'video_completion_rate', 'days_since_last_login']
test_data = val_df[feature_cols].head(5)
csv_payload = test_data.to_csv(index=False)
print('Sending CSV payload:')
print(csv_payload[:300])

runtime_client = boto3.client('sagemaker-runtime')
try:
    response = runtime_client.invoke_endpoint(
        EndpointName='student-completion-endpoint-fixed',
        ContentType='text/csv',
        Body=csv_payload
    )
    result = response['Body'].read().decode('utf-8')
    print(f'Success! Predictions: {result}')
except Exception as e:
    print(f'Failed: {e}')

In [ ]:
import concurrent.futures, threading
runtime = boto3.client('sagemaker-runtime')
errors, successes = 0, 0
lock = threading.Lock()

def invoke(payload):
    global errors, successes
    try:
        runtime.invoke_endpoint(EndpointName='student-completion-endpoint-fixed', ContentType='text/csv', Body=payload)
        with lock: successes += 1
    except Exception as e:
        with lock: errors += 1

# Use explicit feature columns for test payload
feature_cols = ['courses_enrolled', 'courses_completed_prev', 'avg_time_per_module',
                'login_frequency', 'forum_posts', 'assignment_submission_rate',
                'video_completion_rate', 'days_since_last_login']
payload = val_df[feature_cols].head(10).to_csv(index=False)
print('CSV Payload preview:')
print(payload[:200])
print('\nStarting load test with 200 requests across 20 threads...')
print('This will trigger aggressive scaling and potentially cause errors.\n')

start_time = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as ex:
    futures = [ex.submit(invoke, payload) for _ in range(200)]
    for i, f in enumerate(concurrent.futures.as_completed(futures)):
        if (i+1) % 50 == 0:
            print(f'Progress: {i+1}/200 - Success: {successes}, Errors: {errors}')

elapsed = time.time() - start_time
print(f'\nLoad test completed in {elapsed:.1f}s')
print(f'Success: {successes}, Errors: {errors}')
print(f'Error Rate: {errors/(successes+errors)*100:.1f}%')
print(f'Throughput: {200/elapsed:.1f} requests/sec')

In [ ]:
autoscaling = boto3.client('application-autoscaling')
sm_client = boto3.client('sagemaker')
resource_id = f'endpoint/student-completion-endpoint-fixed/variant/AllTraffic'

autoscaling.register_scalable_target(
    ServiceNamespace='sagemaker',
    ResourceId=resource_id,
    ScalableDimension='sagemaker:variant:DesiredInstanceCount',
    MinCapacity=1,
    MaxCapacity=4
)

# BUG: TargetValue=10 is too low, ScaleOutCooldown=0 causes thrashing
autoscaling.put_scaling_policy(
    PolicyName='bad-scaling-policy',
    ServiceNamespace='sagemaker',
    ResourceId=resource_id,
    ScalableDimension='sagemaker:variant:DesiredInstanceCount',
    PolicyType='TargetTrackingScaling',
    TargetTrackingScalingPolicyConfiguration={
        'TargetValue': 10.0,  # BUG: Too low!
        'PredefinedMetricSpecification': {'PredefinedMetricType': 'SageMakerVariantInvocationsPerInstance'},
        'ScaleInCooldown': 600,
        'ScaleOutCooldown': 0  # BUG: No cooldown!
    }
)
print('Aggressive scaling configured!')

## Part 6: load testing the behavious of the SageMaker endpoint with misconfigured autoscaling policy


In [ ]:
import concurrent.futures, threading, time
from collections import defaultdict

# Initialize clients
runtime = boto3.client('sagemaker-runtime')
sm_client = boto3.client('sagemaker')
autoscaling = boto3.client('application-autoscaling')
resource_id = 'endpoint/student-completion-endpoint-fixed/variant/AllTraffic'

stats = defaultdict(int)
lock = threading.Lock()

# Prepare CSV payload with explicit test data
# Generate fresh test data to avoid any variable scope issues
feature_cols = ['courses_enrolled', 'courses_completed_prev', 'avg_time_per_module',
                'login_frequency', 'forum_posts', 'assignment_submission_rate',
                'video_completion_rate', 'days_since_last_login']

# Create sample test data directly
test_rows = []
for i in range(10):
    test_rows.append({
        'courses_enrolled': 5,
        'courses_completed_prev': 3,
        'avg_time_per_module': 45.5,
        'login_frequency': 7.2,
        'forum_posts': 12,
        'assignment_submission_rate': 0.85,
        'video_completion_rate': 0.72,
        'days_since_last_login': 5
    })
test_df = pd.DataFrame(test_rows)
payload = test_df.to_csv(index=False)

print('='*60)
print('LOAD TEST - Triggering Auto-Scaling Chaos')
print('='*60)
print(f'Target: student-completion-endpoint')
print(f'Scaling Policy: TargetValue=10 (too low!), Cooldown=0 (thrashing!)')
print()
print('CSV Payload:')
print(payload[:200])
print()

# Validate endpoint works before load test
print('Validating endpoint before load test...')
try:
    resp = runtime.invoke_endpoint(
        EndpointName='student-completion-endpoint-fixed',
        ContentType='text/csv',
        Body=payload
    )
    result = resp['Body'].read().decode('utf-8')
    print(f'Validation successful! Sample prediction: {result[:50]}')
except Exception as e:
    print(f'ERROR: Endpoint validation failed: {e}')
    print('Fix the endpoint before running load test!')
    raise

def invoke(payload):
    try:
        runtime.invoke_endpoint(
            EndpointName='student-completion-endpoint-fixed',
            ContentType='text/csv',
            Body=payload
        )
        with lock: stats['success'] += 1
    except Exception as e:
        error_type = type(e).__name__
        with lock:
            stats['errors'] += 1
            stats[f'error_{error_type}'] += 1

print()
# Phase 1: Initial burst to trigger scaling
print('Phase 1: Initial burst (5000 requests, 200 threads)')
print('-'*40)
start = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=200) as ex:
    futures = [ex.submit(invoke, payload) for _ in range(5000)]
    concurrent.futures.wait(futures)
print(f'  Completed in {time.time()-start:.1f}s')
print(f'  Success: {stats["success"]}, Errors: {stats["errors"]}')

# Check scaling activity
print('\nChecking scaling activity...')
try:
    activities = autoscaling.describe_scaling_activities(
        ServiceNamespace='sagemaker', ResourceId=resource_id, MaxResults=3
    )['ScalingActivities']
    for a in activities:
        print(f'  {a["StatusCode"]}: {a["Description"][:80]}')
except Exception as e:
    print(f'  Could not get scaling activities: {e}')

# Phase 2: Sustained heavy load during scaling
print('\nPhase 2: Sustained heavy load (15000 requests, 500 threads)')
print('-'*40)
print('This may cause errors as instances scale up/down...')
start = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=500) as ex:
    futures = [ex.submit(invoke, payload) for _ in range(15000)]
    for i, f in enumerate(concurrent.futures.as_completed(futures)):
        if (i+1) % 3000 == 0:
            print(f'  Progress: {i+1}/15000 - Success: {stats["success"]}, Errors: {stats["errors"]}')
print(f'  Completed in {time.time()-start:.1f}s')

# Phase 3: Massive spike to overwhelm
print('\nPhase 3: Traffic spike (10000 requests, 1000 threads)')
print('-'*40)
start = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=1000) as ex:
    futures = [ex.submit(invoke, payload) for _ in range(10000)]
    concurrent.futures.wait(futures)
print(f'  Completed in {time.time()-start:.1f}s')

# Final results
print('\n' + '='*60)
print('LOAD TEST RESULTS')
print('='*60)
total = stats['success'] + stats['errors']
print(f'Total Requests: {total}')
print(f'Successful: {stats["success"]} ({stats["success"]/total*100:.1f}%)')
print(f'Failed: {stats["errors"]} ({stats["errors"]/total*100:.1f}%)')
print('\nError Breakdown:')
for k, v in stats.items():
    if k.startswith('error_'):
        print(f'  {k.replace("error_", "")}: {v}')

# Check endpoint health
print('\nEndpoint Status:')
ep = sm_client.describe_endpoint(EndpointName='student-completion-endpoint-fixed')
print(f'  Status: {ep["EndpointStatus"]}')
for v in ep.get('ProductionVariants', []):
    print(f'  Instances: {v.get("CurrentInstanceCount", "N/A")}')

---

# 🚨 Scenario 3: The Auto-Scaling Chaos
## Status: Investigation Required + Cost Alert

**The Alert:**
```
ALARM: Abnormal Auto-Scaling Activity Detected
Endpoint: student-completion-predictor-endpoint
Instance Count: Oscillating rapidly
CloudWatch Alarms: Flooding (800+ notifications)
Cost Alert: Projected significant monthly overage
```

**The Situation:**
The endpoint is now healthy, but something is terribly wrong with auto-scaling. Instances are scaling up and down rapidly, creating a "thrashing" pattern. CloudWatch is flooding with alarms. The AWS bill is climbing fast.

**The Impact:**
- Significant cost overrun from aggressive scaling
- Unstable service due to constant instance changes
- Alarm fatigue from hundreds of notifications
- Potential budget exhaustion

**Your Challenge:**
The auto-scaling configuration is causing chaos. Find the misconfiguration before the monthly budget is blown.

**Traditional Approach:** Extended time understanding behavior, reviewing policies, and recalculating thresholds.

**With AWS DevOps Agent:** Let's see how it identifies the root cause and recommends optimal settings...

---

## Part 7 Now Let us troubleshoot the Auto Scaling issue of the SageMaker Endpoint using AWS DevOps Agent.
#### Navigate to the DevOps Agents Incident Response Dashboard and start the investigation by providing the following details about the issue you are observing.

###### Investigation Details: ```we are seeing model invocation errors on the student-completion-endpoint-fixed SageMaker Endpoint in our account```

###### Investigation Starting point : ```Please review the model invocation logs and the respective endpoint related configurations to identify the root cause of the issue.```


In [ ]:
# Delete bad policy
try:
    autoscaling.delete_scaling_policy(
        PolicyName='bad-scaling-policy',
        ServiceNamespace='sagemaker',
        ResourceId=resource_id,
        ScalableDimension='sagemaker:variant:DesiredInstanceCount'
    )
except: pass

# Create fixed policy
autoscaling.put_scaling_policy(
    PolicyName='fixed-scaling-policy',
    ServiceNamespace='sagemaker',
    ResourceId=resource_id,
    ScalableDimension='sagemaker:variant:DesiredInstanceCount',
    PolicyType='TargetTrackingScaling',
    TargetTrackingScalingPolicyConfiguration={
        'TargetValue': 1000.0,  # FIXED: Reasonable target
        'PredefinedMetricSpecification': {'PredefinedMetricType': 'SageMakerVariantInvocationsPerInstance'},
        'ScaleInCooldown': 300,
        'ScaleOutCooldown': 60  # FIXED: Proper cooldown
    }
)
print('Fixed scaling policy applied!')

---

# 🌅 Morning Debrief: Lessons Learned
## Status: ALL SCENARIOS RESOLVED

**Workshop Review:**

You just completed three troubleshooting scenarios:
1. **Training Job Failure** (KeyError) - Resolved rapidly
2. **Unhealthy Endpoint** (Missing health check + memory leak) - Resolved rapidly  
3. **Auto-Scaling Chaos** (Misconfigured policies) - Resolved rapidly

**Total Resolution Time:** A fraction of what traditional troubleshooting would require.

---

## �� Key Takeaways

### Technical Lessons:
- **Data validation matters:** Schema changes break pipelines
- **SageMaker requirements are strict:** Health checks, model loading, inference format
- **Memory management is critical:** Unbounded growth crashes containers
- **Auto-scaling needs tuning:** Default settings rarely match your workload
- **Logs tell stories:** But only if you can parse them quickly

### Operational Lessons:
- **Speed matters:** Every minute of downtime has business impact
- **Correlation is key:** Issues often have multiple root causes
- **Automation wins:** Manual log searching doesn't scale
- **Proactive monitoring:** Detect issues before customers do

### Business Impact:
- **Reduced MTTR:** From hours to minutes
- **Cost savings:** Prevented wasted compute and scaling overruns
- **Better operations:** Engineers don't need to be heroes at 2 AM
- **Customer satisfaction:** Faster resolution means less disruption
- **Innovation time:** Less firefighting, more building

---

## 🎯 What You've Learned

You now understand:
- How ML workflows fail in production
- What AWS DevOps Agent can detect and analyze
- How to investigate incidents using AI-powered tools
- Best practices for SageMaker training and inference
- Cost optimization strategies for auto-scaling

**Most importantly:** You've experienced the difference between reactive firefighting and proactive, AI-powered incident response.

---

*Now let's clean up the resources and wrap up this workshop.*

---

## Part 8: Cleanup

In [ ]:
def cleanup():
    resource_id = 'endpoint/student-completion-endpoint/variant/AllTraffic'
    for policy in ['bad-scaling-policy', 'fixed-scaling-policy']:
        try:
            autoscaling.delete_scaling_policy(PolicyName=policy, ServiceNamespace='sagemaker',
                ResourceId=resource_id, ScalableDimension='sagemaker:variant:DesiredInstanceCount')
        except: pass
    try:
        autoscaling.deregister_scalable_target(ServiceNamespace='sagemaker',
            ResourceId=resource_id, ScalableDimension='sagemaker:variant:DesiredInstanceCount')
    except: pass
    try: sm_client.delete_endpoint(EndpointName='student-completion-endpoint')
    except: pass
    try: sm_client.delete_endpoint_config(EndpointConfigName='student-completion-endpoint')
    except: pass
    for m in sm_client.list_models(NameContains='student-completion').get('Models', []):
        try: sm_client.delete_model(ModelName=m['ModelName'])
        except: pass
    print('Cleanup complete!')

# Uncomment to run the cleanup function:
# cleanup()